# 📈 Real-Time Stock News Producer (Kafka)

This Jupyter Notebook acts as the streaming data generator for our real-time financial sentiment analysis pipeline. It simulates realistic financial news events for major tech and market-moving tickers and publishes them to an Apache Kafka cluster running in KRaft mode.

---

### 🔍 What this Producer does:
1. Connects to the local Apache Kafka broker listening on `localhost:9092`.
2. Inspects Kafka cluster topics and creates the `stock-news` topic if it does not already exist.
3. Simulates realistic market headlines across major tickers (`AAPL`, `TSLA`, `NVDA`, `MSFT`, `AMZN`, `GOOGL`, `META`, `AMD`, `JPM`, `BAC`).
4. Generates structured JSON payloads containing:
   - **`ticker`**: Stock symbol (e.g., `AAPL`, `TSLA`, `NVDA`)
   - **`headline`**: Realistic financial news headline (earnings beats, AI product launches, regulatory checks, etc.)
   - **`timestamp`**: Standard ISO 8601 UTC timestamp string
5. Publishes a new message to the `stock-news` topic every **2 seconds** in an infinite loop.
6. Gracefully handles manual interruptions (`KeyboardInterrupt`), ensuring Kafka producer buffers are flushed and connections are cleanly terminated.

---

### 🚀 How to Run in JupyterLab:
1. Make sure your Docker Kafka container is active (`docker compose up -d`).
2. Verify that the notebook kernel is set to **`Python (Stock Sentiment)`** (or your active `.venv`).
3. Select the code cell below and click the **Play (Run)** button or press `Shift + Enter`.
4. Observe the live stream of generated stock news printed in the cell output.

---

### ⏹️ How to Stop Safely:
- Click the **Square Stop Button ("Interrupt the kernel")** on the JupyterLab toolbar.
- Or navigate to the top menu: **Kernel -> Interrupt Kernel**.
- The `try ... except KeyboardInterrupt` block will catch the interrupt signal, flush any pending messages, close the producer, and exit cleanly.

In [ ]:
import os
import sys
import json
import time
import random
import urllib.request
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from pymongo import MongoClient
from kafka import KafkaProducer
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError, KafkaError

# ==========================================
# Configuration
# ==========================================
BOOTSTRAP_SERVERS = 'localhost:9092'
TOPIC_NAME = 'stock-news'
PUBLISH_INTERVAL_SECONDS = 2
MAX_MESSAGES = int(os.getenv('MAX_PRODUCER_MESSAGES', '0'))

def resolve_mongo_uri():
    if os.getenv('MONGO_URI'):
        return os.getenv('MONGO_URI')
    secrets_file = os.path.join('.streamlit', 'secrets.toml')
    if os.path.exists(secrets_file):
        try:
            with open(secrets_file, 'r') as f:
                for line in f:
                    if line.strip().startswith('MONGO_URI'):
                        return line.split('=', 1)[1].strip().strip('"').strip("'")
        except Exception:
            pass
    return 'mongodb://localhost:27017/'

MONGO_URI = resolve_mongo_uri()
MONGO_DB = 'StockDB'
LOGS_COLLECTION = 'pipeline_logs'

TICKERS = ["AAPL", "TSLA", "NVDA", "MSFT", "AMZN", "GOOGL", "META", "AMD", "JPM", "BAC"]

# Realistic market event templates as fallback
HEADLINE_TEMPLATES = [
    "{ticker} reports quarterly earnings and revenue beating Wall Street estimates by a wide margin.",
    "{ticker} unveils next-generation enterprise AI accelerator chips, boosting forward guidance.",
    "Analyst upgrades {ticker} to Strong Buy with an aggressive price target increase.",
    "{ticker} announces record quarterly free cash flow and a new $15 billion share buyback program.",
    "Major European regulatory body greenlights {ticker}'s landmark multi-billion dollar acquisition.",
    "{ticker} forms strategic cloud computing and robotics alliance with global automotive leaders.",
    "{ticker} faces new antitrust investigation from regulators over alleged market monopolization.",
    "{ticker} cuts full-year revenue outlook citing prolonged semiconductor supply chain constraints.",
    "Disappointing quarterly results: {ticker} misses EPS forecast as operating expenses rise sharply.",
    "{ticker} shares tumble following executive departures and product delivery delays.",
    "{ticker} schedules its Q3 fiscal financial results conference call and webcast for next week.",
    "{ticker} concludes annual general shareholder meeting and ratifies board appointments.",
    "{ticker} trades sideways in quiet trading session ahead of Federal Reserve interest rate decision."
]

def fetch_yahoo_headline(ticker):
    """Fetch genuine real-world breaking news from Yahoo Finance RSS."""
    try:
        url = f'https://finance.yahoo.com/rss/headline?s={ticker}'
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=3) as resp:
            xml_data = resp.read()
        root = ET.fromstring(xml_data)
        items = root.findall('.//item')
        if items:
            title = items[0].find('title')
            if title is not None and title.text:
                return title.text.strip()
    except Exception:
        pass
    return random.choice(HEADLINE_TEMPLATES).format(ticker=ticker)

def ensure_kafka_topic(bootstrap_servers, topic_name):
    """Verify or create the Kafka topic using KafkaAdminClient."""
    print(f'[Producer] Checking Kafka broker at {bootstrap_servers}...')
    try:
        admin_client = KafkaAdminClient(
            bootstrap_servers=bootstrap_servers,
            client_id='stock_news_admin',
            request_timeout_ms=5000
        )
        existing_topics = admin_client.list_topics()
        if topic_name in existing_topics:
            print(f"[Admin] Topic '{topic_name}' already exists.")
        else:
            print(f"[Admin] Topic '{topic_name}' not found. Creating topic...")
            new_topic = NewTopic(name=topic_name, num_partitions=1, replication_factor=1)
            admin_client.create_topics(new_topics=[new_topic], validate_only=False)
            print(f"[Admin] Topic '{topic_name}' created successfully.")
        admin_client.close()
    except TopicAlreadyExistsError:
        print(f"[Admin] Topic '{topic_name}' already exists.")
    except Exception as e:
        print(f"[Admin] Topic check status: {e}")

def run_producer():
    """Main publishing loop generating and streaming stock news."""
    ensure_kafka_topic(BOOTSTRAP_SERVERS, TOPIC_NAME)

    # Connect to MongoDB for logging (optional)
    logs_col = None
    try:
        mongo_client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
        mongo_client.admin.command('ping')
        logs_col = mongo_client[MONGO_DB][LOGS_COLLECTION]
        print(f'[Producer] Connected to MongoDB logs at {MONGO_URI.split("@")[-1]}')
    except Exception:
        print('[Producer] MongoDB not connected; continuing without remote logging.')

    print(f'[Producer] Initializing KafkaProducer for {BOOTSTRAP_SERVERS}...')
    try:
        producer = KafkaProducer(
            bootstrap_servers=BOOTSTRAP_SERVERS,
            value_serializer=lambda v: json.dumps(v).encode('utf-8'),
            acks='all',
            retries=3
        )
        print(f"[Producer] Connected successfully. Publishing to '{TOPIC_NAME}' every {PUBLISH_INTERVAL_SECONDS}s.")
        print('=' * 80)
    except Exception as e:
        print(f'[Producer Error] Failed to connect to Kafka at {BOOTSTRAP_SERVERS}: {e}')
        return

    message_count = 0
    try:
        while True:
            ticker = random.choice(TICKERS)
            headline = fetch_yahoo_headline(ticker)
            timestamp = datetime.now(timezone.utc).isoformat()

            payload = {
                'ticker': ticker,
                'headline': headline,
                'timestamp': timestamp
            }

            producer.send(TOPIC_NAME, value=payload)
            producer.flush()

            message_count += 1
            now_str = datetime.now(timezone.utc).strftime('%H:%M:%S.%f')[:-3]
            print(f'[{message_count:04d}] Sent -> {ticker}: "{headline}" | {timestamp}')

            # Log to MongoDB Atlas pipeline_logs (visible live on App dashboard)
            if logs_col is not None:
                try:
                    logs_col.insert_one({
                        'time': now_str,
                        'iso_time': timestamp,
                        'level': 'PRODUCER',
                        'component': 'KAFKA_PRODUCER',
                        'message': f'Sent -> {ticker}: "{headline[:55]}..."'
                    })
                except Exception:
                    pass

            if MAX_MESSAGES > 0 and message_count >= MAX_MESSAGES:
                print(f'[Producer] Completed {MAX_MESSAGES} messages. Exiting cleanly.')
                break

            time.sleep(PUBLISH_INTERVAL_SECONDS)

    except KeyboardInterrupt:
        print('\n[Producer] KeyboardInterrupt received. Stopping producer loop safely...')
    except Exception as err:
        print(f'\n[Producer Error] Unexpected exception: {err}')
    finally:
        print('[Producer] Closing Kafka producer...')
        producer.flush()
        producer.close()
        print('[Producer] Shutdown complete. Resources released cleanly.')

run_producer()
